# 📓 Notebook Prático — Spark ML com a base real NYC TLC (Unidade 5)

**Processamento de Dados Massivos** — Ciência de Dados e Inteligência Artificial

Neste notebook, percorremos o ciclo completo de um projeto de aprendizado de máquina com **Apache Spark**, usando dados reais de corridas de táxi de Nova York (2024):

1. **Ambiente:** como o Spark se organiza no Colab, e o que muda em um cluster.
2. **Paralelismo:** o efeito do número de partições no tempo de processamento.
3. **Qualidade dos dados:** valores ausentes, inconsistências lógicas, outliers e duplicatas.
4. **Engenharia de atributos:** variáveis temporais, duração da corrida e integração com outra fonte (join).
5. **Pipeline de pré-processamento:** indexação, one-hot encoding, vetorização e padronização, sem vazamento de dados.
6. **Modelagem supervisionada:** classificação (tarifa acima ou abaixo da mediana) e regressão (valor da tarifa).
7. **Modelagem não supervisionada:** clusterização das corridas com K-Means.

> **Como usar:** execute as células em ordem (**Ambiente de execução → Executar tudo**, ou `Ctrl+F9`). Algumas células dependem de variáveis criadas nas anteriores.

---
## 1. Ambiente e dados

### 1.1 Criando a sessão Spark

A `SparkSession` é o ponto de entrada de qualquer programa Spark: é por ela que lemos dados, criamos DataFrames e configuramos a execução.

No Colab, o Spark roda em **modo local** (`local[*]`). Isso quer dizer que o *driver* (o processo que coordena o trabalho) e os *executores* (que processam os dados) ficam todos na mesma máquina virtual, e o `*` indica que todos os núcleos de CPU disponíveis serão usados. O código é **o mesmo** que rodaria em um cluster; muda apenas o endereço do `master` (veja a seção 2.5).

O Colab já traz o PySpark instalado. A biblioteca `datasets`, do Hugging Face, é usada para baixar a base.

In [ ]:
!pip install -q datasets

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("Unidade5_ML_NYCTaxi_Real")
         .master("local[*]")
         .getOrCreate())

print("Versão do Spark:", spark.version)

### 1.2 Carregando os dados

Vamos usar duas fontes:

- **`taxi_zone_lookup.csv`:** tabela oficial da TLC que traduz o código numérico de cada zona da cidade (`LocationID`) no nome do bairro (*Borough*) e da zona. Vamos usá-la na seção 4 para enriquecer os dados com um *join*.
- **Amostra de 100 mil corridas de 2024:** hospedada no Hugging Face, extraída dos arquivos oficiais da TLC.

In [ ]:
!wget -q -nc https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

In [ ]:
from datasets import load_dataset

ds = load_dataset("alexvaroz/nyc_tripdata_2024_sample_100k", split="train")
df = spark.createDataFrame(ds.to_pandas())

O caminho dos dados foi: Hugging Face → pandas (na memória do driver) → DataFrame Spark. Para 100 mil linhas, isso funciona bem. Para bases realmente grandes, o correto seria o Spark ler os arquivos **diretamente** (`spark.read.parquet(...)`), de forma distribuída, sem passar tudo pela memória de um único processo.

Vejamos o esquema, isto é, as colunas e seus tipos:

In [ ]:
df.printSchema()

As colunas mais importantes para este notebook:

| Coluna | Significado |
|---|---|
| `tpep_pickup_datetime` / `tpep_dropoff_datetime` | Data e hora do embarque e do desembarque |
| `passenger_count` | Número de passageiros, informado pelo motorista |
| `trip_distance` | Distância percorrida, em milhas |
| `PULocationID` / `DOLocationID` | Zona de embarque e de desembarque (códigos da `taxi_zone_lookup`) |
| `payment_type` | Forma de pagamento: 1 = cartão de crédito, 2 = dinheiro, 3 = sem cobrança, 4 = disputa, ... |
| `fare_amount` | Tarifa calculada pelo taxímetro (tempo + distância), **sem** gorjeta, taxas e pedágios |
| `total_amount` | Valor total cobrado do passageiro |

A variável que vamos prever é a **`fare_amount`**.

In [ ]:
df.count()

### 1.3 Valores ausentes: `NaN` não é `null`

Há uma armadilha na conversão pandas → Spark. No pandas, um valor ausente em coluna numérica é representado como **`NaN`** (*Not a Number*). Ao converter, o Spark mantém esse `NaN` como um **número de ponto flutuante**, e não como **`null`**, que é o valor ausente "oficial" do Spark.

Na prática, isso faz diferença:

- `isNull()` **não** encontra os `NaN`, então a contagem de ausentes dá zero.
- Qualquer média envolvendo `NaN` resulta em `NaN`.

Vamos comparar as duas contagens:

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, isnan

colunas_double = [c for c, tipo in df.dtypes if tipo == "double"]

df.select(
    [F.sum(col(c).isNull().cast("int")).alias(c) for c in colunas_double]
).show(vertical=True)

In [ ]:
df.select(
    [F.sum(isnan(col(c)).cast("int")).alias(c) for c in colunas_double]
).show(vertical=True)

A primeira tabela (`null`) mostra zeros, mas a segunda (`NaN`) revela os ausentes de verdade. Vamos padronizar: todo `NaN` das colunas numéricas vira `null`. A partir daqui, as funções do Spark tratam os ausentes corretamente: `describe()` e `mean()` os ignoram, e `isNull()` os encontra.

In [ ]:
for c in colunas_double:
    df = df.withColumn(c, F.when(isnan(col(c)), None).otherwise(col(c)))

df.select(F.sum(col("passenger_count").isNull().cast("int")).alias("passenger_count_nulos")).show()

---
## 2. Paralelismo: núcleos, partições e desempenho

Um DataFrame Spark é dividido em **partições**, que são pedaços dos dados. Cada partição é processada por uma **tarefa** (*task*), e cada núcleo de CPU executa uma tarefa por vez. Vamos responder a três perguntas:

- Quantos núcleos o Colab oferece?
- Em quantas partições os dados foram divididos?
- O número de partições influencia o desempenho?

### 2.1 Inspecionando o ambiente

In [ ]:
spark

In [ ]:
sc = spark.sparkContext
print("Paralelismo padrão (núcleos disponíveis):", sc.defaultParallelism)

O `defaultParallelism` indica quantas tarefas podem rodar **simultaneamente**. No Colab gratuito, normalmente são **2 núcleos**, então no máximo 2 tarefas em paralelo.

Agora, o número de partições do DataFrame:

In [ ]:
df.rdd.getNumPartitions()

Como o DataFrame foi criado a partir de dados locais (pandas), o Spark o dividiu em `defaultParallelism` partições, uma por núcleo. Vejamos quantas linhas ficaram em cada uma:

In [ ]:
from pyspark.sql.functions import spark_partition_id

df.groupBy(spark_partition_id().alias("particao")).count().orderBy("particao").show()

As partições estão equilibradas. Isso é importante: se uma partição fosse muito maior que as outras (*data skew*), a tarefa responsável por ela atrasaria todo o job.

### 2.2 Experimento: partições de *shuffle* × tempo

Operações como `groupBy` e `join` exigem um **shuffle**: os dados são redistribuídos entre as partições para que todas as linhas com a mesma chave (aqui, o mesmo `payment_type`) fiquem juntas. O número de partições **depois** do shuffle é controlado por `spark.sql.shuffle.partitions`, cujo padrão é **200**.

O experimento a seguir:

1. Faz uma agregação (corridas e receita por tipo de pagamento) com diferentes valores de `spark.sql.shuffle.partitions`.
2. Mede o tempo de cada configuração 3 vezes e usa a **mediana**, que é menos sensível a oscilações que a média.

O DataFrame base vai para o cache (`cache()`), para que o tempo medido seja o do processamento, e não o da leitura dos dados.

In [ ]:
df_base = df.select("payment_type", "total_amount").cache()
df_base.count()  # ação que materializa o cache

In [ ]:
import time


def consulta(d):
    return d.groupBy("payment_type").agg(
        F.count("*").alias("corridas"),
        F.sum("total_amount").alias("receita")
    )


def medir(n_part, repeticoes=3):
    """Executa a agregação com n_part partições de shuffle e devolve a mediana do tempo."""
    spark.conf.set("spark.sql.shuffle.partitions", str(n_part))
    tempos = []
    for _ in range(repeticoes):
        inicio = time.perf_counter()
        resultado = consulta(df_base)
        resultado.collect()  # ação: dispara a execução
        tempos.append(time.perf_counter() - inicio)
    mediana = sorted(tempos)[len(tempos) // 2]
    return mediana

Repare no `collect()`: o Spark é **preguiçoso** (*lazy evaluation*). As transformações (`groupBy`, `agg`) apenas montam um plano de execução, e nada é processado até que uma **ação** (`collect`, `count`, `show`, `write`...) seja chamada. Sem a ação, estaríamos medindo só o tempo de montar o plano.

In [ ]:
import pandas as pd

valores = [1, 2, 4, 8, 16, 50, 200, 1000]
resultados = []
for n in valores:
    t = medir(n)
    resultados.append({"shuffle_partitions": n, "tempo_s": round(t, 3)})
    print(f"{n:>5} partições -> {t:.3f} s")

tabela = pd.DataFrame(resultados)
tabela

### 2.3 Visualizando o resultado

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(tabela["shuffle_partitions"], tabela["tempo_s"], marker="o")
plt.xscale("log")
plt.xticks(valores, [str(v) for v in valores])
plt.axvline(sc.defaultParallelism, linestyle="--", color="gray",
            label=f"nº de núcleos ({sc.defaultParallelism})")
plt.xlabel("spark.sql.shuffle.partitions (escala log)")
plt.ylabel("Tempo da agregação (s)")
plt.title("Tempo do groupBy x número de partições de shuffle")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Como interpretar:**

- **Poucas partições, próximo ao número de núcleos:** melhores tempos. Todos os núcleos trabalham, e há pouco custo de coordenação.
- **Muitas partições (200, 1000):** o tempo cresce. Cada partição vira uma tarefa, e cada tarefa tem um custo fixo de agendamento, serialização e escrita do shuffle. Com 100 mil linhas e só 4 valores distintos de `payment_type`, a maior parte das 1000 tarefas processa **nada**, mas o custo de cada uma é pago mesmo assim.
- **Regra prática:** usar algumas vezes o número de núcleos (2 a 4 partições por núcleo), ajustando conforme o volume. O padrão de 200 foi pensado para clusters e volumes grandes, e costuma ser excessivo para bases pequenas.

Com base no experimento, vamos fixar **4 partições** de shuffle (2 por núcleo):

In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "4")

### 2.4 Partições de leitura × partições de shuffle

São **duas configurações diferentes**, que costumam confundir:

| | Partições de leitura | Partições de shuffle |
|---|---|---|
| O que define | Como o dado de entrada é dividido | Quantas partições existem **depois** de um `groupBy` ou `join` |
| Controlado por | Fonte de dados / `defaultParallelism` / `repartition()` | `spark.sql.shuffle.partitions` |

Por isso, mudar `spark.sql.shuffle.partitions` **não** altera o DataFrame original:

In [ ]:
print("Partições do df (leitura):", df.rdd.getNumPartitions())  # continua 2

Já o resultado de um `groupBy` deveria ter 4 partições. Vejamos:

In [ ]:
print("Partições após groupBy:", df.groupBy("payment_type").count().rdd.getNumPartitions())

O resultado provavelmente foi **1**, e não 4. O responsável é o **AQE** (*Adaptive Query Execution*), ligado por padrão desde o Spark 3.2. Durante a execução, o AQE observa o tamanho real dos dados e **junta partições de shuffle pequenas**, para evitar justamente o desperdício visto no experimento. Como o resultado tem só alguns registros, tudo coube em uma partição.

Desligando o AQE, o valor configurado é respeitado:

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
print("AQE desligado:", df.groupBy("payment_type").count().rdd.getNumPartitions())  # 4

Desligamos o AQE apenas para observar o efeito. Em uso real, ele deve ficar **ligado**, então vamos religá-lo:

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")

### 2.5 Do Colab para um cluster

Para rodar este mesmo código em um cluster Spark Standalone, como o montado no laboratório com Docker, basta apontar o `master` para o endereço do cluster e, se quiser, definir os recursos de cada executor:

```python
spark = (SparkSession.builder
         .master("spark://spark-master:7077")      # endereço do cluster
         .config("spark.executor.cores", "1")      # núcleos por executor
         .config("spark.executor.memory", "512m")  # memória por executor
         .getOrCreate())
```

**Atenção:** isso precisa ser feito **antes** de criar a primeira sessão. O `getOrCreate()` devolve a sessão que já existe, se houver uma, e nesse caso ignora `master` e as configurações de executor. Por isso, executar esse código aqui no Colab, depois da seção 1, não conectaria a cluster nenhum. Além disso, o endereço `spark-master` só existe dentro da rede do laboratório.

---
## 3. Qualidade dos dados

Antes de modelar, precisamos conhecer e tratar os problemas da base. Um modelo treinado com dados ruins aprende padrões ruins (*garbage in, garbage out*).

### 3.1 Estatísticas descritivas

In [ ]:
df.select("trip_distance", "fare_amount", "total_amount",
          "passenger_count").describe().show()

Pontos que chamam atenção:

- **`count` de `passenger_count` menor que 100.000:** são os ausentes que convertemos de `NaN` para `null` na seção 1.3.
- **Valores mínimos negativos ou zero** em `fare_amount`, `total_amount` e `trip_distance`: tarifas negativas costumam ser estornos, e distância zero indica corridas canceladas ou erro do taxímetro.
- **Máximos muito altos:** podem ser erros de digitação ou corridas excepcionais.

### 3.2 Valores ausentes nas colunas-chave

In [ ]:
cols_chave = ["passenger_count", "trip_distance", "fare_amount",
              "payment_type", "tpep_pickup_datetime"]
df.select([F.sum(col(c).isNull().cast("int")).alias(c) for c in cols_chave]).show()

Só `passenger_count` tem ausentes. Vamos tratá-los com **imputação** na seção 5, dentro do pipeline.

### 3.3 Testes de consistência lógica

Além dos ausentes, há registros **preenchidos, mas impossíveis**. Testamos regras que o mundo real impõe:

- **Distância:** uma corrida precisa percorrer distância maior que zero.
- **Tarifa mínima:** a bandeirada de um táxi amarelo de NYC é **$3,00** desde dez/2022 (era $2,50). Tarifas abaixo disso indicam estorno ou erro.
- **Tarifas absurdas:** valores acima de $300 são raríssimos, e normalmente são erros.
- **Passageiros:** corridas com 0 passageiro.
- **Horários:** o desembarque não pode ocorrer antes do embarque.

In [ ]:
total = df.count()

problemas = {
    "Distância zero ou negativa": df.filter(col("trip_distance") <= 0).count(),
    "Tarifa negativa ou zero":    df.filter(col("fare_amount") <= 0).count(),
    "Tarifas absurdas (> $300)":  df.filter(col("fare_amount") > 300).count(),
    "Passageiros = 0":            df.filter(col("passenger_count") == 0).count(),
    "Dropoff antes do pickup":    df.filter(
        col("tpep_dropoff_datetime") < col("tpep_pickup_datetime")).count(),
}

for nome, qtd in problemas.items():
    print(f"{nome}: {qtd:,} ({qtd/total*100:.2f}%)")

### 3.4 Aplicando os filtros

Removemos os registros inconsistentes e restringimos o estudo aos pagamentos em **cartão (1)** e **dinheiro (2)**, que são a grande maioria. Os demais códigos (sem cobrança, disputa, anulada) representam situações atípicas, em que a tarifa não reflete uma corrida normal.

Usamos $2,50 como piso, valor da tarifa mínima antiga. Ele é um pouco mais permissivo que a bandeirada atual, de $3,00.

As corridas com **0 passageiro** foram mantidas: o campo é digitado pelo motorista e costuma vir zerado por esquecimento, sem que a corrida em si seja inválida.

In [ ]:
df_limpo = (df
    .filter(col("trip_distance") > 0)
    .filter(col("fare_amount") >= 2.50)              # tarifa mínima
    .filter(col("fare_amount") <= 300)               # remove casos extremos
    .filter(col("tpep_dropoff_datetime") > col("tpep_pickup_datetime"))
    .filter(col("payment_type").isin([1, 2]))        # apenas cartão e dinheiro
)

print(f"Original: {total:,} | Após filtros: {df_limpo.count():,}")

### 3.5 Outliers pelo método do IQR

O **intervalo interquartil** (IQR = Q3 − Q1) define limites estatísticos para valores "típicos":

$$[\,Q1 - 1{,}5 \cdot IQR,\;\; Q3 + 1{,}5 \cdot IQR\,]$$

Usamos `approxQuantile`, que calcula os quartis de forma **aproximada** (erro relativo de 1%). Isso faz diferença em bases massivas: o quantil exato exige ordenar todos os dados, o que é caro em um ambiente distribuído.

In [ ]:
q1, q3 = df_limpo.approxQuantile("fare_amount", [0.25, 0.75], 0.01)
iqr = q3 - q1
lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Q1 = {q1:.2f} | Q3 = {q3:.2f} | IQR = {iqr:.2f}")
print(f"Limites aceitáveis: [{lim_inf:.2f}, {lim_sup:.2f}]")

In [ ]:
acima = df_limpo.filter(col("fare_amount") > lim_sup).count()
print(f"Corridas acima do limite superior: {acima:,} "
      f"({acima / df_limpo.count() * 100:.1f}% da base limpa)")

Mais de 10% das corridas seriam classificadas como *outliers*. Mas elas **não são erros**: são principalmente corridas **longas**, como as idas aos aeroportos. A corrida entre Manhattan e o aeroporto JFK, por exemplo, tem **tarifa fixa** de $70. Como 75% das corridas são curtas e baratas, o limite estatístico fica baixo.

Remover essas corridas tornaria o modelo **incapaz de prever tarifas altas**. Por isso, o filtro abaixo fica **comentado**: o IQR aponta *candidatos* a outliers, mas a decisão de remover é de **negócio**, e não apenas estatística.

In [ ]:
# Não aplicado — ver justificativa acima
# df_limpo = df_limpo.filter(
#     (col("fare_amount") >= lim_inf) & (col("fare_amount") <= lim_sup))

### 3.6 Média ou mediana para preencher `passenger_count`?

Para escolher a estratégia de imputação, comparamos as duas medidas, ignorando os ausentes:

In [ ]:
df_limpo.select(
    F.mean("passenger_count").alias("media_passageiros"),
    F.median("passenger_count").alias("mediana_passageiros"),
).show()

A distribuição é **assimétrica**: a grande maioria das corridas tem 1 passageiro, e poucas têm 5 ou 6. A média é puxada para cima e resulta em um valor fracionário, como 1,3 passageiro, que não existe. Já a **mediana**, igual a 1, é robusta a esses extremos e corresponde a um valor real. Por isso, usaremos `strategy="median"` no `Imputer`.

**Por que não imputar agora?** Os parâmetros da imputação (a mediana) precisam ser calculados **somente com os dados de treino**. Se calculássemos com a base inteira, informações do conjunto de teste "vazariam" para o treino (*data leakage*), e a avaliação ficaria otimista. Por isso, o `Imputer` entrará no **pipeline** da seção 5, que é ajustado apenas no treino.

### 3.7 Duplicatas

In [ ]:
antes = df_limpo.count()
df_limpo = df_limpo.dropDuplicates()
print(f"Duplicatas exatas removidas: {antes - df_limpo.count():,}")

df_limpo.cache()  # evita recomputar toda a limpeza nas próximas etapas

Nenhuma duplicata exata. Mesmo assim, vale verificar sempre: falhas na ingestão costumam gerar registros repetidos.

O `cache()` guarda o DataFrame limpo em memória. Sem ele, cada ação posterior refaria **toda** a cadeia de transformações desde a leitura, por causa da avaliação preguiçosa.

---
## 4. Engenharia de atributos

A qualidade de um modelo depende menos do algoritmo e mais dos **atributos** que o alimentam. Nesta etapa, extraímos estrutura preditiva dos campos brutos.

- **Atributos temporais:**
  - `pickup_hour`: hora do embarque;
  - `pickup_dow`: dia da semana, de 1 = domingo a 7 = sábado;
  - `eh_fim_de_semana`: indica sábado ou domingo.

  Eles capturam padrões de demanda e de trânsito, como horários de pico, madrugadas e fins de semana.
- **Duração da corrida (`trip_duration_min`):** diferença entre desembarque e embarque, em minutos. Costuma ser um dos melhores preditores da tarifa, porque o taxímetro de NYC cobra por distância **e** por tempo parado no trânsito. Duas corridas de mesma distância podem ter tarifas bem diferentes. É um atributo que só existe porque **combinamos duas colunas**.
- **Custo por milha (`custo_por_milha`):** razões entre variáveis expõem relações que o valor absoluto esconde. É útil para **análise exploratória**, por exemplo para achar corridas com cobrança anormal.

> ⚠️ **Cuidado com o vazamento:** `custo_por_milha` é calculado **a partir da própria `fare_amount`**, que é a variável que queremos prever. Usá-la como atributo de entrada seria "colar" a resposta no modelo. Por isso, ela **não** entra no pipeline de modelagem.

Por fim, criamos `payment_label`, um rótulo textual para o tipo de pagamento ("credito" ou "dinheiro"), que será codificado na seção 5.

In [ ]:
from itertools import chain
from pyspark.sql.functions import (hour, dayofweek, when, round as _round,
                                   unix_timestamp, create_map, lit)

# Mapeamento dos códigos oficiais da TLC
mapa_pagamento = {1: "credito", 2: "dinheiro"}
mapping_expr = create_map(*[lit(x) for x in
    chain.from_iterable(mapa_pagamento.items())])

df_feat = (df_limpo
    .withColumn("trip_duration_min",
        _round((unix_timestamp("tpep_dropoff_datetime")
              - unix_timestamp("tpep_pickup_datetime")) / 60, 2))
    .withColumn("pickup_hour", hour("tpep_pickup_datetime"))
    .withColumn("pickup_dow", dayofweek("tpep_pickup_datetime"))
    .withColumn("eh_fim_de_semana",
        when(dayofweek("tpep_pickup_datetime").isin([1, 7]), 1.0).otherwise(0.0))
    .withColumn("custo_por_milha",
        _round(col("fare_amount") / col("trip_distance"), 2))
    .withColumn("payment_label", mapping_expr[col("payment_type").cast("int")])
)

df_feat.select("trip_distance", "trip_duration_min", "pickup_hour",
               "eh_fim_de_semana", "payment_label", "fare_amount").show(5)

### 4.1 Integração de fontes (join)

O `PULocationID` é apenas um código numérico. O *join* com a tabela `taxi_zone_lookup` traduz cada código no bairro de embarque (`pickup_borough`). É o cenário real de combinar fontes distintas por uma **chave de ligação**.

Usamos um **left join** para manter todas as corridas, mesmo que algum código não exista na tabela de zonas. Nesse caso, o bairro fica `null`.

Como a tabela de zonas é pequena (265 linhas), o Spark aplica automaticamente um **broadcast join**: ele envia uma cópia dela para cada executor, em vez de fazer shuffle da tabela grande.

In [ ]:
zonas = spark.read.csv("taxi_zone_lookup.csv", header=True, inferSchema=True)
zonas.select("LocationID", "Borough", "Zone").show(5)

df_feat = (df_feat
    .join(zonas.select("LocationID", "Borough")
           .withColumnRenamed("Borough", "pickup_borough"),
          df_feat.PULocationID == zonas.LocationID, "left")
    .drop("LocationID")
)

df_feat.groupBy("pickup_borough").count().orderBy("count", ascending=False).show()

Manhattan concentra a grande maioria dos embarques de táxis amarelos, seguido pelo Queens, onde ficam os aeroportos JFK e LaGuardia. Neste notebook, o bairro é usado apenas para **análise**. Como exercício, experimente incluí-lo como atributo categórico no pipeline: ele precisaria de um `StringIndexer` e de um `OneHotEncoder`, como o tipo de pagamento.

---
## 5. Divisão treino/teste e pipeline de pré-processamento

### 5.1 Divisão treino/teste

Separamos **80% para treino** e **20% para teste** com `randomSplit`:

- **Conjunto de treino:** ensina o modelo.
- **Conjunto de teste:** avalia a capacidade do modelo de **generalizar** para dados que ele nunca viu.

A semente (`seed=42`) garante reprodutibilidade: a mesma divisão em execuções diferentes.

In [ ]:
train_df, test_df = df_feat.randomSplit([0.8, 0.2], seed=42)
print(f"Treino: {train_df.count():,} | Teste: {test_df.count():,}")

A divisão não dá exatamente 80/20, porque o `randomSplit` sorteia **cada linha** de forma independente, com probabilidade 0,8. Isso evita ordenar ou contar a base inteira antes de dividir, o que seria caro em dados massivos.

### 5.2 Montando o pipeline

Os algoritmos do Spark ML esperam **uma única coluna** de entrada, `features`, do tipo **vetor**. O pipeline transforma as colunas da tabela nesse vetor, em etapas:

| Etapa | O que faz | Por quê |
|---|---|---|
| `Imputer` | Preenche `passenger_count` ausente com a mediana do treino → `passengers` | Os algoritmos não aceitam valores ausentes |
| `StringIndexer` | "credito"/"dinheiro" → 0.0/1.0 (`payment_idx`) | Algoritmos só operam sobre números |
| `OneHotEncoder` | Índice → vetor binário (`payment_vec`) | Evita que o modelo interprete uma ordem inexistente entre as categorias |
| `VectorAssembler` | Junta as colunas numéricas em um vetor (`features_num`) | Formato exigido pelo Spark ML |
| `StandardScaler` | Padroniza: média 0 e desvio 1 (`features_num_scaled`) | Coloca atributos de escalas diferentes (milhas, minutos, horas) em pé de igualdade |
| `VectorAssembler` final | Junta numéricas padronizadas + categóricas → `features` | Vetor final de entrada do modelo |

A padronização é feita **apenas nas colunas numéricas**. O vetor one-hot já está em 0/1, e padronizá-lo só atrapalharia a interpretação.

Há um detalhe no `OneHotEncoder`: com 2 categorias, ele gera um vetor de **tamanho 1**. Por padrão (`dropLast=True`), a última categoria é representada por "tudo zero", o que evita redundância entre as colunas.

**Ponto central:** o pipeline é ajustado (`fit`) **somente no treino**. Mediana, índices das categorias, média e desvio-padrão vêm exclusivamente dos dados de treino, e depois são **aplicados** (`transform`) igualmente ao treino e ao teste. Assim, o teste simula de verdade dados novos.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import (Imputer, StringIndexer, OneHotEncoder,
                                VectorAssembler, StandardScaler)

num_cols = ["trip_distance", "trip_duration_min", "passengers",
            "pickup_hour", "eh_fim_de_semana"]

imputer = Imputer(inputCols=["passenger_count"], outputCols=["passengers"],
                  strategy="median")
indexer = StringIndexer(inputCol="payment_label", outputCol="payment_idx",
                        handleInvalid="skip")
encoder = OneHotEncoder(inputCol="payment_idx", outputCol="payment_vec")
assembler_num = VectorAssembler(inputCols=num_cols, outputCol="features_num")
scaler = StandardScaler(inputCol="features_num",
                        outputCol="features_num_scaled",
                        withMean=True, withStd=True)
assembler_final = VectorAssembler(
    inputCols=["features_num_scaled", "payment_vec"], outputCol="features")

pipeline = Pipeline(stages=[imputer, indexer, encoder, assembler_num,
                            scaler, assembler_final])
pipeline_model = pipeline.fit(train_df)   # fit APENAS no treino

train_prep = pipeline_model.transform(train_df)
test_prep = pipeline_model.transform(test_df)

train_prep.select("passengers", "payment_label", "payment_vec", "features").show(5, truncate=False)

Repare na diferença entre os dois objetos:

- **`pipeline`** (um `Pipeline`) é só a **receita**: a lista de etapas, sem nada aprendido.
- **`pipeline_model`** (um `PipelineModel`) é o resultado do `fit`: guarda o que foi **aprendido** no treino, como a mediana e a média e o desvio de cada coluna.

### 5.3 Salvando e recarregando o pipeline

O `PipelineModel` pode ser salvo em disco e recarregado depois, em outro programa ou em outro momento, para transformar novos dados **exatamente** da mesma forma. É assim que se leva um pré-processamento para produção.

In [ ]:
pipeline_model.write().overwrite().save("/tmp/pipeline_nyc_real")

In [ ]:
from pyspark.ml import PipelineModel

PipelineModel.load("/tmp/pipeline_nyc_real") \
    .transform(test_df.limit(5)) \
    .select("features", "fare_amount").show(5, truncate=False)

---
## 6. Classificação: a tarifa ficará acima da mediana?

Para praticar **classificação binária**, transformamos a tarifa em uma pergunta de sim ou não:

- `label = 1`: tarifa **acima** da mediana (corrida "cara");
- `label = 0`: tarifa até a mediana (corrida "barata").

Usar a mediana como corte gera **classes balanceadas**, com cerca de 50% em cada uma. Isso simplifica a avaliação, porque um classificador que "chuta" sempre a mesma classe acertaria só 50%.

> **Observação:** para manter a coerência com a seção 5, o ideal seria calcular a mediana apenas no treino. Aqui a calculamos sobre `df_feat` por simplicidade. Como a mediana de 80% dos dados é praticamente igual à de 100%, o efeito é desprezível.

In [ ]:
mediana = df_feat.approxQuantile("fare_amount", [0.5], 0.01)[0]
print(f"Tarifa mediana real: ${mediana:.2f}")

df_cls = df_feat.withColumn(
    "label", when(col("fare_amount") > mediana, 1.0).otherwise(0.0))

train_c, test_c = df_cls.randomSplit([0.8, 0.2], seed=42)
train_c_prep = pipeline_model.transform(train_c)
test_c_prep = pipeline_model.transform(test_c)

train_c_prep.groupBy("label").count().show()

Reaproveitamos o `pipeline_model` já ajustado: o pré-processamento é o mesmo, e só o alvo mudou.

### 6.1 Treinando dois classificadores

- **Regressão Logística:** modelo **linear**, que estima a probabilidade da classe 1. É simples, rápido e interpretável.
- **Random Forest:** conjunto de **árvores de decisão**, cada uma treinada com uma amostra diferente dos dados e dos atributos, que votam na classe final. Captura relações não lineares e interações entre atributos.

In [ ]:
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier

modelo_lr = LogisticRegression(featuresCol="features", labelCol="label",
                               maxIter=1000).fit(train_c_prep)

In [ ]:
modelo_rf = RandomForestClassifier(featuresCol="features", labelCol="label",
                                   numTrees=50, seed=42).fit(train_c_prep)

In [ ]:
pred_lr = modelo_lr.transform(test_c_prep)
pred_rf = modelo_rf.transform(test_c_prep)

pred_lr.select("fare_amount", "label", "probability", "prediction").show(5, truncate=False)

O `transform` adiciona três colunas:

- `rawPrediction`: pontuação bruta de cada classe;
- `probability`: probabilidade de cada classe;
- `prediction`: classe prevista, a de maior probabilidade, ou seja, corte em 0,5.

### 6.2 Avaliação

| Métrica | O que mede |
|---|---|
| **AUC** (área sob a curva ROC) | Capacidade de **ordenar**: a chance de o modelo dar probabilidade maior a uma corrida cara do que a uma barata. Vai de 0,5 (aleatório) a 1 (perfeito) e não depende do corte de 0,5 |
| **Precisão** | Das corridas que o modelo disse serem de uma classe, quantas realmente eram |
| **Recall** | Das corridas que eram de uma classe, quantas o modelo encontrou |
| **F1** | Média harmônica de precisão e recall |

As versões *weighted* calculam cada métrica por classe e fazem a média ponderada pelo tamanho de cada classe.

In [ ]:
from pyspark.ml.evaluation import (BinaryClassificationEvaluator,
                                   MulticlassClassificationEvaluator)

evs = {
    "AUC":      BinaryClassificationEvaluator(labelCol="label",
                                              metricName="areaUnderROC"),
    "F1":       MulticlassClassificationEvaluator(labelCol="label",
                                                  metricName="f1"),
    "Precisão": MulticlassClassificationEvaluator(labelCol="label",
                                                  metricName="weightedPrecision"),
    "Recall":   MulticlassClassificationEvaluator(labelCol="label",
                                                  metricName="weightedRecall"),
}

for nome, pred in [("Logistic Regression", pred_lr),
                   ("Random Forest", pred_rf)]:
    print(f"{nome}: " + " | ".join(
        f"{m}={e.evaluate(pred):.3f}" for m, e in evs.items()))

Os dois modelos devem ter desempenho alto. Isso é esperado: distância e duração determinam quase diretamente a tarifa, já que o taxímetro é calculado a partir delas. Os erros se concentram nas corridas com tarifa **perto da mediana**, em que uma pequena diferença de distância ou de tempo muda a classe.

### 6.3 (Opcional) Ajuste de hiperparâmetros com validação cruzada

**Hiperparâmetros** são escolhas feitas antes do treino, como a força da regularização (`regParam`) ou o número de iterações (`maxIter`). Para escolhê-los sem "espiar" o conjunto de teste, usamos **validação cruzada** sobre o treino:

1. O `ParamGridBuilder` define a **grade** de combinações. Aqui são 3 × 2 = 6 combinações.
2. O `CrossValidator` divide o treino em `numFolds=3` partes. Para cada combinação, treina com 2 partes e avalia na terceira, revezando as partes. Isso dá **18 treinos** no total.
3. A combinação de melhor métrica média é escolhida e retreinada com todo o treino.

O parâmetro `parallelism=4` treina até 4 modelos ao mesmo tempo. É uma etapa **demorada** no Colab; para executá-la, mude `EXECUTAR_CV` para `True`.

In [ ]:
EXECUTAR_CV = False

if EXECUTAR_CV:
    from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
    import numpy as np

    lr_base = LogisticRegression(featuresCol="features", labelCol="label")
    grid = (ParamGridBuilder()
            .addGrid(lr_base.regParam, [0.01, 0.1, 1.0])
            .addGrid(lr_base.maxIter, [10, 50])
            .build())

    cv = CrossValidator(estimator=lr_base, estimatorParamMaps=grid,
                        evaluator=BinaryClassificationEvaluator(labelCol="label"),
                        numFolds=3, parallelism=4, seed=42)
    cv_model = cv.fit(train_c_prep)

    melhor_idx = int(np.argmax(cv_model.avgMetrics))
    print("Melhores parâmetros:", cv.getEstimatorParamMaps()[melhor_idx])
    print(f"Melhor AUC médio: {max(cv_model.avgMetrics):.3f}")
    print(f"AUC no teste: {evs['AUC'].evaluate(cv_model.transform(test_c_prep)):.3f}")

---
## 7. Regressão: prevendo o valor da tarifa

Agora o alvo é a própria `fare_amount`, um **valor contínuo** em dólares. Usamos `train_prep` e `test_prep`, preparados na seção 5.

| Métrica | O que mede | Unidade |
|---|---|---|
| **RMSE** (raiz do erro quadrático médio) | Erro típico, penalizando mais os erros grandes | dólares |
| **MAE** (erro absoluto médio) | Erro médio, em valor absoluto | dólares |
| **R²** (coeficiente de determinação) | Fração da variação da tarifa explicada pelo modelo; 1 é perfeito e 0 equivale a prever sempre a média | — |

### 7.1 Regressão Linear

In [ ]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

lr_reg = LinearRegression(featuresCol="features", labelCol="fare_amount")
pred_reg = lr_reg.fit(train_prep).transform(test_prep)

for metrica in ["rmse", "mae", "r2"]:
    ev = RegressionEvaluator(labelCol="fare_amount",
                             predictionCol="prediction",
                             metricName=metrica)
    print(f"{metrica.upper():4}: {ev.evaluate(pred_reg):.3f}")

pred_reg.select("fare_amount", "prediction").show(10)

Quando o RMSE é bem maior que o MAE, há **erros grandes concentrados** em poucas corridas. O RMSE eleva cada erro ao quadrado, então poucos erros grandes pesam muito. Uma causa provável são as corridas de **tarifa fixa** (aeroportos), cuja tarifa não segue a relação linear com distância e tempo.

### 7.2 Modelos baseados em árvores

- **Random Forest:** média de muitas árvores independentes (*bagging*), que reduz a variância.
- **GBT** (*Gradient-Boosted Trees*): árvores treinadas **em sequência**, cada uma corrigindo os erros das anteriores (*boosting*). Costuma ter o melhor desempenho em dados tabulares, mas é mais lento, porque as árvores não podem ser treinadas em paralelo entre si.

Como árvores dividem o espaço em faixas, elas conseguem capturar relações **não lineares**, como as tarifas fixas e os efeitos de horário.

In [ ]:
from pyspark.ml.regression import RandomForestRegressor, GBTRegressor

ev = RegressionEvaluator(labelCol="fare_amount",
                         predictionCol="prediction", metricName="rmse")

rf_reg = RandomForestRegressor(featuresCol="features",
                               labelCol="fare_amount",
                               numTrees=50, seed=42).fit(train_prep)
print(f"Random Forest RMSE: {ev.evaluate(rf_reg.transform(test_prep)):.3f}")

In [ ]:
gbt = GBTRegressor(featuresCol="features", labelCol="fare_amount",
                   maxIter=50, seed=42).fit(train_prep)
print(f"GBT RMSE: {ev.evaluate(gbt.transform(test_prep)):.3f}")

### 7.3 Quais atributos mais influenciam a tarifa?

Modelos de árvore informam a **importância** de cada atributo: quanto ele contribuiu, somando todas as árvores, para reduzir o erro nas divisões.

A ordem dos nomes abaixo segue a ordem do vetor `features`: primeiro as 5 colunas numéricas, depois a única posição de `payment_vec`.

In [ ]:
imp = pd.DataFrame({"feature": num_cols + ["payment_vec"],
                    "importance": rf_reg.featureImportances.toArray()})
print(imp.sort_values("importance", ascending=False).to_string(index=False))

Distância e duração devem dominar, o que é coerente com o funcionamento do taxímetro, que cobra por milha e por minuto parado. Os atributos temporais e o tipo de pagamento contribuem pouco para a **tarifa** em si.

> Importância **não é causalidade**. Ela mede o quanto o modelo **usou** o atributo, e atributos correlacionados, como distância e duração, dividem a importância entre si.

---
## 8. Clusterização: agrupando corridas parecidas

Até aqui, sempre havia uma resposta conhecida para aprender. Na **clusterização**, que é aprendizado **não supervisionado**, não há rótulo: o algoritmo procura grupos de corridas **semelhantes entre si**.

O **K-Means**:

1. Escolhe `k` centros iniciais.
2. Atribui cada corrida ao centro mais próximo.
3. Recalcula cada centro como a média das corridas do grupo.
4. Repete os passos 2 e 3 até estabilizar.

Como o K-Means usa **distância euclidiana**, a padronização é obrigatória. Sem ela, a duração em minutos dominaria o número de passageiros.

Usamos distância, duração, tarifa e passageiros. Aqui a tarifa **pode** entrar, porque não estamos prevendo nada. O `Imputer` entra de novo no pipeline, porque `passenger_count` tem ausentes.

In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

imputer_km = Imputer(inputCols=["passenger_count"], outputCols=["passengers"],
                     strategy="median")
assembler_km = VectorAssembler(
    inputCols=["trip_distance", "trip_duration_min", "fare_amount", "passengers"],
    outputCol="features_km_raw")
scaler_km = StandardScaler(inputCol="features_km_raw",
                           outputCol="features", withMean=True, withStd=True)

df_km = Pipeline(stages=[imputer_km, assembler_km, scaler_km]) \
    .fit(df_feat).transform(df_feat).cache()

modelo_km = KMeans(featuresCol="features", predictionCol="cluster",
                   k=4, seed=42).fit(df_km)
resultado = modelo_km.transform(df_km)

### 8.1 Quantos clusters?

O número `k` precisa ser escolhido. Testamos de 2 a 8 com dois critérios:

- **Custo** (*inertia*): soma das distâncias ao quadrado de cada ponto ao seu centro. Sempre **diminui** com mais clusters. Procuramos o "**cotovelo**", ou seja, o ponto a partir do qual aumentar `k` traz pouco ganho.
- **Silhouette:** de −1 a 1, mede se cada ponto está mais perto do próprio cluster do que do vizinho. **Quanto maior, melhor**, e valores acima de 0,5 indicam grupos bem separados.

In [ ]:
avaliador = ClusteringEvaluator(featuresCol="features")

for k in range(2, 9):
    m = KMeans(k=k, seed=42, featuresCol="features").fit(df_km)
    sil = avaliador.evaluate(m.transform(df_km))
    print(f"K={k} | Custo={m.summary.trainingCost:14.2f} | Silhouette={sil:.4f}")

### 8.2 Interpretando os clusters

Os centros estão em escala padronizada, então são difíceis de ler. Para interpretar os grupos, olhamos as **médias em valores originais** de cada cluster e o tamanho de cada um:

In [ ]:
resultado.groupBy("cluster").agg(
    F.count("*").alias("corridas"),
    _round(F.avg("trip_distance"), 2).alias("distancia_media_mi"),
    _round(F.avg("trip_duration_min"), 1).alias("duracao_media_min"),
    _round(F.avg("fare_amount"), 2).alias("tarifa_media"),
    _round(F.avg("passengers"), 2).alias("passageiros_medio"),
).orderBy("cluster").show()

Tipicamente, aparecem grupos como:

- **corridas curtas urbanas**, a maioria;
- **corridas médias**;
- **corridas longas**, como as de aeroporto, com tarifa alta;
- um grupo separado pelo **número de passageiros**.

Dar nome aos clusters é uma etapa **humana**: o algoritmo só agrupa, e a interpretação depende do conhecimento do domínio.

---
## 9. Para discussão

1. No experimento da seção 2, por que 1000 partições foram tão mais lentas, se o volume de dados era o mesmo? O que mudaria em uma base de 1 bilhão de linhas em um cluster com 100 núcleos?
2. Por que o `Imputer` e o `StandardScaler` precisam ser ajustados **só no treino**? Dê um exemplo de como o vazamento poderia inflar as métricas.
3. O IQR apontou mais de 10% das corridas como outliers. Que informação de negócio justifica mantê-las? Em que situação seria correto removê-las?
4. Por que `custo_por_milha` não pode ser usado como atributo para prever `fare_amount`?
5. Compare RMSE e MAE dos três regressores. O que a diferença entre as duas métricas revela sobre a distribuição dos erros?
6. Que atributos você acrescentaria para melhorar a previsão das corridas de tarifa fixa, como as de aeroporto? *Dica: `PULocationID`, `DOLocationID` e `RatecodeID`.*

In [2]:
ministros_presentes = "Rodrigo Pacheco, Odair Cunha e Antonio Anastasia"
print(ministros_presentes.split())

['Rodrigo', 'Pacheco,', 'Odair', 'Cunha', 'e', 'Antonio', 'Anastasia']


In [1]:
ministros_presentes = "Rodrigo Pacheco, Odair Cunha e Antonio Anastasia"
print(ministros_presentes.split())

['Rodrigo', 'Pacheco,', 'Odair', 'Cunha', 'e', 'Antonio', 'Anastasia']
